In [6]:
import numpy as np

from factoralign.constraints import (
    fully_invested,
    box_bounds,
    active_bounds,
    factor_active_bounds,
)

n_assets = 5

benchmark = np.array([0.20, 0.20, 0.20, 0.20, 0.20])

style_exposures = np.array([
    [1.2,  0.4],
    [0.7, -0.1],
    [-0.2, 1.0],
    [0.3,  0.5],
    [-0.5, -0.8],
])

constraints = (
    fully_invested(n_assets)
    .combine(box_bounds(0.0, 0.40, n_assets))
    .combine(active_bounds(benchmark, -0.10, 0.10))
    .combine(
        factor_active_bounds(
            exposures=style_exposures,
            benchmark=benchmark,
            lower_active=np.array([-0.05, -0.03]),
            upper_active=np.array([0.05, 0.03]),
            factor_names=["value", "momentum"],
        )
    )
)

print("n_assets:", constraints.n_assets)
print("n_ineq:", constraints.n_ineq)
print("n_eq:", constraints.n_eq)

print("\nG shape:", constraints.G.shape)
print("u shape:", constraints.u.shape)

print("\nC:")
print(constraints.C)

print("\nd:")
print(constraints.d)

n_assets: 5
n_ineq: 24
n_eq: 1

G shape: (24, 5)
u shape: (24,)

C:
[[1. 1. 1. 1. 1.]]

d:
[1.]


In [7]:
import numpy as np

from factoralign.projection import project_onto_factors

X = np.array([
    [1.0, 0.2],
    [1.0, 0.5],
    [1.0, 1.0],
    [1.0, 1.5],
])

z = np.array([0.03, 0.02, -0.01, 0.04])
specific_var = np.array([0.04, 0.03, 0.05, 0.02])
weights = 1.0 / specific_var

result = project_onto_factors(z, X, weights=weights)

print(result.spanned)
print(result.orthogonal)
print(result.coefficients)
print(result.orthogonality_error(X, weights))

[0.017947   0.02095843 0.02597749 0.03099655]
[ 0.012053   -0.00095843 -0.03597749  0.00900345]
[0.01593937 0.01003812]
8.473409486550036e-16


In [9]:
import numpy as np

from factoralign.constraints import fully_invested, box_bounds
from factoralign.optimizer import solve_mvo, kkt_residual

n_assets = 5

alpha = np.array([0.04, 0.02, 0.03, -0.08, 0.12])
Q = np.eye(n_assets) * 0.05

constraints = (
    fully_invested(n_assets)
    .combine(box_bounds(0.0, 0.40, n_assets))
)

result = solve_mvo(
    alpha=alpha,
    covariance=Q,
    constraints=constraints,
    risk_aversion=5.0,
)

print("holdings:")
print(result.holdings)

print("\ninequality duals:")
print(result.ineq_duals)

print("\nequality duals:")
print(result.eq_duals)

print("\nslack:")
print(constraints.slack(result.holdings))

print("\nbinding constraints:")
binding = constraints.binding_mask(result.holdings)
for name, dual, slack in zip(constraints.names, result.ineq_duals, constraints.slack(result.holdings)):
    if abs(dual) > 1e-8 or slack < 1e-8:
        print(f"{name:20s} dual={dual:.6f}, slack={slack:.6e}")

residual = kkt_residual(alpha, Q, constraints, result, risk_aversion=5.0)
print("\nKKT error:")
print(np.linalg.norm(residual))

holdings:
[2.4000000e-01 1.6000000e-01 2.0000000e-01 2.7364449e-23 4.0000000e-01]

inequality duals:
[0.   0.   0.   0.   0.04 0.   0.   0.   0.06 0.  ]

equality duals:
[-0.02]

slack:
[1.6000000e-01 2.4000000e-01 2.0000000e-01 4.0000000e-01 0.0000000e+00
 2.4000000e-01 1.6000000e-01 2.0000000e-01 2.7364449e-23 4.0000000e-01]

binding constraints:
upper_bound_4        dual=0.040000, slack=0.000000e+00
lower_bound_3        dual=0.060000, slack=2.736445e-23

KKT error:
1.5899003276961254e-17


In [10]:
import numpy as np

from factoralign.alignment import analyze_alignment, orthogonal_norm_share
from factoralign.constraints import fully_invested, box_bounds
from factoralign.optimizer import solve_mvo
from factoralign.risk import RiskModel

n_assets = 5

alpha = np.array([0.04, 0.02, 0.03, -0.08, 0.12])

X = np.array([
    [1.0,  0.2],
    [1.0,  0.5],
    [1.0,  1.0],
    [1.0, -0.3],
    [1.0,  1.5],
])

factor_cov = np.diag([0.02, 0.03])
specific_var = np.full(n_assets, 0.05)

risk_model = RiskModel(
    X=X,
    factor_cov=factor_cov,
    specific_var=specific_var,
)

Q = risk_model.covariance()

constraints = (
    fully_invested(n_assets)
    .combine(box_bounds(0.0, 0.40, n_assets))
)

result = solve_mvo(
    alpha=alpha,
    covariance=Q,
    constraints=constraints,
    risk_aversion=5.0,
)

analysis = analyze_alignment(
    alpha=alpha,
    constraints=constraints,
    ineq_duals=result.ineq_duals,
    eq_duals=result.eq_duals,
    exposures=risk_model.X,
    weights=risk_model.inverse_specific_var_weights(),
)

print("alpha orthogonal share:")
print(orthogonal_norm_share(analysis.alpha))

print("ineq pressure orthogonal share:")
print(orthogonal_norm_share(analysis.pressure_ineq))

print("gamma_ineq orthogonal share:")
print(orthogonal_norm_share(analysis.gamma_ineq))

print("gamma_ineq orthogonal direction:")
print(analysis.gamma_direction(include_equalities=False))

alpha orthogonal share:
0.4264443776947169
ineq pressure orthogonal share:
0.0
gamma_ineq orthogonal share:
0.4264443776947169
gamma_ineq orthogonal direction:
[ 0.73799676  0.01907938 -0.51905303 -0.39941592  0.16139281]
